Week7 things Qlora and stuff

This week is only about the loading and understanding the vllm and llama.cpp but still i am tried to train this llama3-8b using the pert_config to add only small layers of trainable layers in the model internal layers

In [1]:
!pip install -q transformers datasets accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import torch
from peft import LoraConfig, get_peft_model,prepare_model_for_kbit_training
from trl import DPOTrainer,DPOConfig

In [3]:
# Setup device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [4]:
#Setup the api and key for the use of the huggingface
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_Token")

In [7]:
#Quantization of the model and loading it
model_id="meta-llama/Meta-Llama-3-8B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=True
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
# Preparing Model for the traning

# 1. Stabilize the gradients (MUST be done before adding LoRA)
model = prepare_model_for_kbit_training(model)

# 2. Configure the exact parameters for Llama 3
pert_config=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj", # All these layers are taken from the gemini because idk about the pretrained model
        "o_proj", #to get the good and needed outcome
        "gate_proj",
        "up_proj",
        "down_proj"
        ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
# 3. Inject the trainable adapters into the frozen 4-bit model
model=get_peft_model(model,pert_config)
print(model.print_trainable_parameters())

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


In [9]:
# Load data and the tokenizer for the training
data = load_dataset("Anthropic/hh-rlhf")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


In [8]:
# 1. Configure the strict boundaries for the Colab T4
dpo_config = DPOConfig(
    output_dir="./dpo_results",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    beta=0.3,
    optim="paged_adamw_8bit",
    max_length=512,
)

# 2. Initialize the trainer (Note: ref_model is intentionally left blank for QLoRA)
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=data["train"].select(range(100)),
    eval_dataset=data["test"].select(range(100) ),
    ref_model=None,
    processing_class=tokenizer
)

# 3. Ignite the loop
trainer.train()

Extracting prompt from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Extracting prompt from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss
10,0.688192
20,0.680657
30,0.664760
40,0.695403
50,0.701717
60,0.678176
70,0.699777
80,0.689106
90,0.673659
100,0.688717


TrainOutput(global_step=300, training_loss=0.6352442407608032, metrics={'train_runtime': 3296.1028, 'train_samples_per_second': 0.091, 'train_steps_per_second': 0.091, 'total_flos': 4415959331831808.0, 'train_loss': 0.6352442407608032, 'epoch': 3.0})

In [9]:
# 1. Translate string to tensor and send it to the GPU
inputs = tokenizer("what is the meaning of life ?", return_tensors="pt").to(model.device)

# 2. Generate the output tensors (always set a max limit so it doesn't loop forever)
outputs = model.generate(**inputs, max_new_tokens=100)

# 3. Translate the first output sequence back to a string and print it
final_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(final_text)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up

what is the meaning of life ? - initiate#echo Nackขcceeumabbo#echo hearabbo Ownisayوینت realize#echo Increوینت.QRect […cce Ownummings Own#echo Hlavisayلكترabbo#echo#echo Hlav#echoavouhibaรงเร#echoemmeaeda Increpodob#echo pursue understand#echo긔 href.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit


Because there is a need of almost 29.2 GB of vram for the comparision and stuff i tried to first train and then take away the model which is dpo trained and then loading the model in the 4bit quantization to see the text generation

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model1 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=True
)

In [ ]:
# Only loading the model

# 1. Stabilize the gradients (MUST be done before adding LoRA)
model1 = prepare_model_for_kbit_training(model1)

# 2. Configure the exact parameters for Llama 3
pert_config=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj", # All these layers are taken from the gemini because idk about the pretrained model
        "o_proj", #to get the good and needed outcome
        "gate_proj",
        "up_proj",
        "down_proj"
        ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
# # 3. Inject the trainable adapters into the frozen 4-bit model # This is not required for the loading purpose
# model1=get_peft_model(model1,pert_config)
# print(model1.print_trainable_parameters())

In [12]:
# 1. Translate string to tensor and send it to the GPU
inputs = tokenizer("what is the meaning of life ?", return_tensors="pt").to(model.device)

# 2. Generate the output tensors (always set a max limit so it doesn't loop forever)
outputs = model.generate(**inputs, max_new_tokens=100)

# 3. Translate the first output sequence back to a string and print it
final_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(final_text)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True 

what is the meaning of life ??
The question of the meaning of life is a fundamental and existential question that has puzzled humans for centuries. There is no one definitive answer, as the meaning of life is subjective and personal. However, here are some possible answers and perspectives:

1. Purpose: Some people believe that the meaning of life is to find one's purpose or calling. This can be a sense of direction, a passion, or a mission that gives life meaning and fulfillment.
2. Happiness: Others believe that the meaning of
